# VeriText - TrOCR Handwriting OCR Fine-Tuning Notebook
### Google Colab GPU Training Workflow
This notebook fine-tunes Microsoft's TrOCR (`microsoft/trocr-small-handwritten` or `microsoft/trocr-base-handwritten`) on handwritten line datasets (IAM handwriting database), evaluates Character Error Rate (CER) and Word Error Rate (WER), and exports fine-tuned weights for direct drop-in integration into VeriText.

In [ ]:
# 1. Environment Setup & Hardware Acceleration
!nvidia-smi
!pip install -q transformers evaluate jiwer datasets accelerate torchvision

In [ ]:
# 2. Imports & Seed Setup
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training device: {device}')

In [ ]:
# 3. Load Model and Processor
MODEL_ID = 'microsoft/trocr-small-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

# Set special tokens for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4
model.to(device)

In [ ]:
# 4. Load Dataset (IAM Handwriting or Custom Pairs)
# We load a curated handwriting lines dataset from HuggingFace Hub
dataset = load_dataset('teklia/IAM-line', split='train[:2000]')
split_data = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_data['train']
val_dataset = split_data['test']

def preprocess_batch(batch):
    images = [img.convert('RGB') for img in batch['image']]
    pixel_values = processor(images, return_tensors='pt').pixel_values
    labels = processor.tokenizer(batch['text'], padding='max_length', max_length=64).input_ids
    # Replace pad token id with -100 to ignore in loss calculation
    labels = [[(l if l != processor.tokenizer.pad_token_id else -100) for l in label] for label in labels]
    return {'pixel_values': pixel_values, 'labels': labels}

train_set = train_dataset.map(preprocess_batch, batched=True, batch_size=8, remove_columns=train_dataset.column_names)
val_set = val_dataset.map(preprocess_batch, batched=True, batch_size=8, remove_columns=val_dataset.column_names)

In [ ]:
# 5. Evaluation Metrics: CER & WER
cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')

def compute_metrics(eval_pred):
    labels_ids = eval_pred.label_ids
    pred_ids = eval_pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': cer, 'wer': wer}

In [ ]:
# 6. Fine-Tuning Execution
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    output_dir='./results_trocr',
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.feature_extractor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_set,
    eval_dataset=val_set,
    data_collator=default_data_collator
)

trainer.train()

In [ ]:
# 7. Save Fine-Tuned Model for VeriText
OUTPUT_DIR = './veritext_trocr_weights'
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
!zip -r veritext_trocr_weights.zip ./veritext_trocr_weights
print(f'Weights saved to {OUTPUT_DIR} and zipped. Download veritext_trocr_weights.zip and extract into backend/app/ml/weights/trocr/')